# Clase 100 — Perceptrón, MLP y backpropagación

La **neurona artificial** es la unidad atómica del Deep Learning. En esta clase
implementamos a mano un **perceptrón** (Rosenblatt, 1957), vemos por qué no puede
aprender **XOR**, cómo el **MLP** lo resuelve apilando capas con no linealidades, y
cómo **backpropagation** (la regla de la cadena) calcula los gradientes.

Requiere: `tensorflow` / `keras`, `numpy`, `scikit-learn` (se ejecuta en Colab con GPU).

## 🧠 Intuición previa

**Backpropagation en una frase:** es la *regla de la cadena* repartiendo el error hacia atrás. La red hace un *forward* (capa por capa hasta la pérdida) y luego un *backward*: el gradiente de la pérdida se propaga de la salida hacia la entrada, y en cada capa se multiplica por la derivada local. Así cada peso recibe *su parte* de la culpa del error total. No hay magia: es la derivada de una composición de funciones, calculada de atrás hacia adelante para reutilizar resultados intermedios.

## 1. Perceptrón a mano: aprende AND/OR (linealmente separables)

In [ ]:
import numpy as np
np.random.seed(42)

def step(z):
    # función escalón: 1 si z >= 0, si no 0
    return (z >= 0).astype(float)

def entrena_perceptron(X, y, epocas=20, eta=0.1):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(epocas):
        for xi, yi in zip(X, y):
            y_pred = step(w @ xi + b)
            error = yi - y_pred                  # regla del perceptrón
            w += eta * error * xi
            b += eta * error
    return w, b

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1]); y_or = np.array([0, 1, 1, 1])

for nombre, y in [("AND", y_and), ("OR", y_or)]:
    w, b = entrena_perceptron(X, y)
    pred = step(X @ w + b)
    print(f"{nombre}: pred={pred.astype(int)} | esperado={y} | acc={(pred == y).mean():.2f}")

## 2. El problema XOR: un perceptrón nunca converge

In [ ]:
y_xor = np.array([0, 1, 1, 0])   # no es linealmente separable
w, b = entrena_perceptron(X, y_xor, epocas=100)
pred = step(X @ w + b)
print("XOR con perceptrón:", pred.astype(int), "| esperado:", y_xor)
print(f"accuracy = {(pred == y_xor).mean():.2f}  ->  se queda estancado en 0.5-0.75")
print("Ninguna recta separa (0,0),(1,1) de (0,1),(1,0): hace falta no linealidad.")

## 3. El MLP resuelve XOR (Keras Sequential)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# 1 capa oculta con activación no lineal + salida sigmoide (probabilidad)
modelo = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
               loss="binary_crossentropy", metrics=["accuracy"])
modelo.fit(X, y_xor, epochs=200, verbose=0)   # 4 puntos: se busca overfit perfecto
prob = modelo.predict(X, verbose=0).ravel()
print("probabilidades:", np.round(prob, 3))
print("predicción:", (prob > 0.5).astype(int), "| esperado:", y_xor)

## 4. Forward pass a mano: MLP de 2 capas

`a⁽¹⁾ = σ(W₁·x + b₁)`, luego `ŷ = σ(W₂·a⁽¹⁾ + b₂)`. Lo calculamos con numpy.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# MLP 2-2-1 con pesos fijos (para inspeccionar el flujo)
W1 = np.array([[0.5, -0.3], [0.8, 0.2]]); b1 = np.array([0.1, -0.2])
W2 = np.array([[0.7], [-0.6]]);            b2 = np.array([0.05])

x = np.array([1.0, 0.0])
z1 = W1.T @ x + b1;  a1 = sigmoid(z1)      # capa oculta
z2 = W2.T @ a1 + b2; y_hat = sigmoid(z2)   # salida
print("a1 (oculta):", np.round(a1, 4))
print("ŷ (salida):", np.round(y_hat, 4))

## 5. Backpropagation a mano vs gradiente numérico

Con loss MSE `L = ½(ŷ - y)²` propagamos `∂L/∂W` por la regla de la cadena y lo
verificamos contra una aproximación numérica por diferencias finitas.

In [ ]:
y_true = 1.0

def forward_loss(W1, b1, W2, b2, x, y):
    a1 = sigmoid(W1.T @ x + b1)
    y_hat = sigmoid(W2.T @ a1 + b2)[0]
    return 0.5 * (y_hat - y) ** 2, a1, y_hat

L, a1, y_hat = forward_loss(W1, b1, W2, b2, x, y_true)

# Backprop analítico para W2 (dL/dW2 = dL/dy_hat * dy_hat/dz2 * dz2/dW2)
dL_dyhat = (y_hat - y_true)
dyhat_dz2 = y_hat * (1 - y_hat)
delta2 = dL_dyhat * dyhat_dz2
grad_W2_analitico = np.outer(a1, delta2)     # forma (2, 1)

# Gradiente numérico (diferencias finitas) sobre W2
eps = 1e-6; grad_W2_num = np.zeros_like(W2)
for i in range(W2.shape[0]):
    Wp = W2.copy(); Wp[i, 0] += eps
    Lp, *_ = forward_loss(W1, b1, Wp, b2, x, y_true)
    grad_W2_num[i, 0] = (Lp - L) / eps

print("grad analítico W2:", np.round(grad_W2_analitico.ravel(), 6))
print("grad numérico  W2:", np.round(grad_W2_num.ravel(), 6))
print("coinciden:", np.allclose(grad_W2_analitico, grad_W2_num, atol=1e-4))

## 6. Autograd: `tf.GradientTape` deriva por nosotros

In [ ]:
import tensorflow as tf

# El mismo gradiente que calculamos a mano, ahora automático
xt = tf.constant([1.0, 0.0], dtype=tf.float32)
W2t = tf.Variable(W2, dtype=tf.float32)
a1t = tf.constant(a1, dtype=tf.float32)
with tf.GradientTape() as tape:
    z2 = tf.reduce_sum(W2t[:, 0] * a1t) + float(b2[0])
    yhat = tf.sigmoid(z2)
    loss = 0.5 * (yhat - y_true) ** 2
grad = tape.gradient(loss, W2t)
print("autograd grad W2:", np.round(grad.numpy().ravel(), 6))
print("Autodiff hace innecesario derivar a mano en modelos arbitrarios.")

## Ejercicios

1. **Perceptrón vs XOR**: mostrá con un `print` de la accuracy por época que el
   perceptrón nunca supera ~0.75 en XOR pero llega a 1.0 en AND/OR.
2. **Decision boundary con `make_moons`**: entrená un MLP `[16, 8]` sobre
   `sklearn.datasets.make_moons(noise=0.2)` y graficá la frontera con un meshgrid.
3. **Sin no linealidad**: cambiá la activación oculta del MLP de XOR a `linear` y
   verificá que ya no puede aprenderlo (accuracy ~0.5).
4. **Backprop de W1**: extendé el cálculo manual de la sección 5 para obtener
   `∂L/∂W1` y comparalo con `tf.GradientTape`.

## Conclusiones

- Un perceptrón solo separa clases **linealmente separables**; XOR necesita una capa oculta.
- La **no linealidad** es esencial: sin activación, N capas colapsan a una transformación lineal.
- El **forward pass** compone capas; el **backward pass** aplica la regla de la cadena.
- **Autograd** (GradientTape / `backward()` / `grad`) calcula gradientes sobre cualquier grafo.
- El **teorema de aproximación universal** garantiza que existe el MLP, no que sea fácil de entrenar.

## ✅ Soluciones de los ejercicios

Ejercicios del README resueltos. El **1** y el **4** son numpy puro y llevan `assert` (verificables sin frameworks). El **2, 3 y 5** usan `tensorflow.keras`; si TF no está instalado se validan por AST. Cada celda es autocontenida.

**Ej. 1 — Perceptrón a mano.** En numpy: aprende AND/OR (linealmente separables) pero **nunca** separa XOR.

In [ ]:
import numpy as np
np.random.seed(0)

def train_perceptron(X, y, epochs=20, lr=0.1):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            pred = 1 if (xi @ w + b) > 0 else 0
            err = yi - pred                 # regla del perceptron
            w += lr * err * xi; b += lr * err
    acc = (((X @ w + b) > 0).astype(int) == y).mean()
    return w, b, acc

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_and = np.array([0, 0, 0, 1]); y_or = np.array([0, 1, 1, 1]); y_xor = np.array([0, 1, 1, 0])
for name, yt in [("AND", y_and), ("OR", y_or), ("XOR", y_xor)]:
    print(f"{name}: accuracy={train_perceptron(X, yt)[2]:.2f}")

assert train_perceptron(X, y_and)[2] == 1.0   # separable
assert train_perceptron(X, y_xor)[2] < 1.0    # NO separable
print("OK: el perceptron separa AND/OR pero nunca XOR (no es linealmente separable).")

**Ej. 2 — MLP con Keras.** Una capa oculta ReLU basta para que el MLP resuelva XOR (accuracy 1.0).

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y = np.array([0, 1, 1, 0], dtype="float32")   # XOR

model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=keras.optimizers.Adam(0.05),
              loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=500, verbose=0)
acc = model.evaluate(X, y, verbose=0)[1]
print(f"XOR accuracy con MLP [8]->1: {acc:.2f}")   # -> 1.00

**Ej. 3 — Decision boundary.** MLP `[16, 8]` sobre `make_moons`; la frontera sale claramente curva.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from tensorflow import keras
from tensorflow.keras import layers

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=200, verbose=0)

xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
zz = model.predict(np.c_[xx.ravel(), yy.ravel()], verbose=0).reshape(xx.shape)
plt.contourf(xx, yy, zz, levels=[0, 0.5, 1], alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=20)
plt.title("Frontera MLP [16, 8] sobre make_moons"); plt.show()

**Ej. 4 — Backprop a mano vs gradiente numérico.** MLP 1→2→1, MSE. El gradiente analítico (regla de la cadena) coincide con el numérico — verificado con `assert`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

x = np.array([0.7]); y = np.array([1.0])
W1 = rng.normal(size=(2, 1)); b1 = np.zeros(2)
W2 = rng.normal(size=(1, 2)); b2 = np.zeros(1)

def forward():
    z1 = W1 @ x + b1; a1 = sigmoid(z1)
    yhat = W2 @ a1 + b2                 # salida lineal
    return a1, yhat

def loss():
    _, yhat = forward()
    return 0.5 * float((yhat - y).item() ** 2)

# --- gradiente analitico: regla de la cadena de la salida hacia la entrada ---
a1, yhat = forward()
dyhat = (yhat - y)                      # dL/dyhat
dW2 = np.outer(dyhat, a1)               # dL/dW2
da1 = W2.T @ dyhat                      # propaga a la capa oculta
dz1 = da1 * a1 * (1 - a1)               # derivada local de sigmoid
dW1 = np.outer(dz1, x)                  # dL/dW1

# --- gradiente numerico (diferencias finitas) para comprobar ---
def numgrad(P, i):
    eps = 1e-6; orig = P.flat[i]
    P.flat[i] = orig + eps; lp = loss()
    P.flat[i] = orig - eps; lm = loss()
    P.flat[i] = orig
    return (lp - lm) / (2 * eps)

ng = np.array([numgrad(W1, i) for i in range(W1.size)]).reshape(W1.shape)
print("dW1 analitico:", dW1.ravel())
print("dW1 numerico :", ng.ravel())
assert np.allclose(dW1, ng, atol=1e-5)
print("OK: backprop == gradiente numerico (regla de la cadena verificada).")

**Ej. 5 — Sin activación no lineal.** Con activaciones `linear`, N capas colapsan a una sola transformación lineal: el MLP ya **no** aprende XOR (accuracy ~0.5).

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y = np.array([0, 1, 1, 0], dtype="float32")

model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="linear"),   # sin no-linealidad
    layers.Dense(1, activation="linear"),
])
model.compile(optimizer=keras.optimizers.Adam(0.05), loss="mse", metrics=["accuracy"])
model.fit(X, y, epochs=500, verbose=0)
acc = model.evaluate(X, y, verbose=0)[1]
print(f"XOR con activaciones lineales: accuracy={acc:.2f}  (se queda en ~0.5)")
print("Apilar capas lineales = una unica transformacion lineal. Sin curvatura, no hay XOR.")